# v1.1 Metric Methodology

This notebook establishes the analytical definitions used by the Seattle Public
Safety Dashboard v1.1 before KPI cards, comparison tables, rankings, and
response-time metrics are implemented.

## Questions

1. How should the current analysis period be compared with the immediately
   preceding equal-length period?
2. What is the correct counting unit for crime totals?
3. How should raw and percentage changes be calculated?
4. How should crime rates per 100,000 residents be calculated?
5. How should neighborhood crime-volume rankings and rank changes work?
6. How should map coverage / unmappable-crime percentages be calculated?
7. What exactly should count as a shooting?
8. What constitutes a qualified CAD response-time observation?
9. Which call priorities should contribute to response-time KPIs?
10. How should neighborhood response-time rankings be calculated?
11. Which unified dashboard controls can legitimately affect each metric?

Production KPI logic should not be implemented until the methodology in this
notebook has been reviewed.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display


# -------------------------------------------------------------------
# Locate repository root.
# -------------------------------------------------------------------

cwd = Path.cwd().resolve()

REPO_ROOT = next(
    (
        path
        for path in [cwd, *cwd.parents]
        if (path / "dashboard").is_dir()
        and (path / "app.py").exists()
    ),
    None,
)

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not locate repository root. "
        "Expected app.py and dashboard/."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")


# -------------------------------------------------------------------
# Crime dashboard imports
# -------------------------------------------------------------------

from dashboard.crime_dashboard_data import (
    load_crime_dashboard_context,
    EVENT_ID_COLUMN as CRIME_EVENT_ID_COLUMN,
    ROW_ID_COLUMN as CRIME_REPORT_ID_COLUMN,
    TIME_COLUMN as CRIME_TIME_COLUMN,
    REPORT_TIME_COLUMN as CRIME_REPORT_TIME_COLUMN,
    CATEGORY_COLUMN as CRIME_CATEGORY_COLUMN,
    SUB_CATEGORY_COLUMN as CRIME_SUBCATEGORY_COLUMN,
    LAT_COL as CRIME_LAT_COLUMN,
    LON_COL as CRIME_LON_COLUMN,
    normalize_neighborhood_name,
)


# -------------------------------------------------------------------
# Calls dashboard imports
# -------------------------------------------------------------------

from dashboard.spd_dashboard_data import (
    load_dashboard_context as load_calls_dashboard_context,
)

from dashboard.spd_config import (
    EVENT_ID_COLUMN as CALL_EVENT_ID_COLUMN,
    ROW_ID_COLUMN as CALL_DISPATCH_ID_COLUMN,
    TIME_COLUMN as CALL_TIME_COLUMN,
    ARRIVAL_TIME_COLUMN as CALL_ARRIVAL_COLUMN,
    LAT_COL as CALL_LAT_COLUMN,
    LON_COL as CALL_LON_COLUMN,
)


# -------------------------------------------------------------------
# Display configuration
# -------------------------------------------------------------------

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)


# -------------------------------------------------------------------
# Helpers
# -------------------------------------------------------------------

def clean_string(series: pd.Series) -> pd.Series:
    return (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )


INVALID_TEXT_VALUES = {
    "",
    "-",
    "unknown",
    "nan",
    "none",
    "<na>",
}


def valid_string_mask(series: pd.Series) -> pd.Series:
    cleaned = clean_string(series)

    return (
        cleaned.notna()
        & ~cleaned.isin(INVALID_TEXT_VALUES)
    )


# -------------------------------------------------------------------
# Load the same contexts used by the dashboards.
# -------------------------------------------------------------------

crime_context = load_crime_dashboard_context()
calls_context = load_calls_dashboard_context()

crime_snapshot = crime_context["df"].copy()
crime = crime_context["valid_time"].copy()

mappable_crime = crime_context["mappable_events"].copy()
population = crime_context["neighborhood_population"].copy()

calls = calls_context["df"].copy()
existing_response_analysis = (
    calls_context["response_analysis"].copy()
)


# -------------------------------------------------------------------
# Normalize dates.
# -------------------------------------------------------------------

crime[CRIME_TIME_COLUMN] = pd.to_datetime(
    crime[CRIME_TIME_COLUMN],
    errors="coerce",
)

crime["analysis_date"] = (
    crime[CRIME_TIME_COLUMN]
    .dt.normalize()
)

crime[CRIME_EVENT_ID_COLUMN] = clean_string(
    crime[CRIME_EVENT_ID_COLUMN]
)

crime[CRIME_REPORT_ID_COLUMN] = clean_string(
    crime[CRIME_REPORT_ID_COLUMN]
)

crime[CRIME_SUBCATEGORY_COLUMN] = clean_string(
    crime[CRIME_SUBCATEGORY_COLUMN]
)

crime["event_importance_bin"] = clean_string(
    crime["event_importance_bin"]
)


calls[CALL_TIME_COLUMN] = pd.to_datetime(
    calls[CALL_TIME_COLUMN],
    errors="coerce",
)

calls[CALL_ARRIVAL_COLUMN] = pd.to_datetime(
    calls[CALL_ARRIVAL_COLUMN],
    errors="coerce",
)

calls[CALL_EVENT_ID_COLUMN] = clean_string(
    calls[CALL_EVENT_ID_COLUMN]
)

# -------------------------------------------------------------------
# Canonical v1.1 crime types.
# -------------------------------------------------------------------

CRIME_TYPE_COLUMN = "event_importance_bin"

CANONICAL_CRIME_TYPES = [
    "crimes against persons",
    "crimes against property",
    "crimes against society / other",
]

observed_crime_types = sorted(
    crime[CRIME_TYPE_COLUMN]
    .dropna()
    .unique()
    .tolist()
)

print()
print("Observed analytical crime types:")
for value in observed_crime_types:
    print(f"  {value}")

unexpected_types = sorted(
    set(observed_crime_types)
    - set(CANONICAL_CRIME_TYPES)
)

if unexpected_types:
    print()
    print("WARNING: Non-canonical crime types remain:")
    for value in unexpected_types:
        print(f"  {value}")
else:
    print()
    print("PASS: All observed crime types are canonical v1.1 bins.")

In [2]:
print("POST-CLASSIFICATION CRIME POPULATION")
print("=" * 60)

print(
    f"Unique offenses: "
    f"{crime[CRIME_EVENT_ID_COLUMN].nunique():,}"
)

print(
    f"Unique reports:  "
    f"{crime[CRIME_REPORT_ID_COLUMN].nunique():,}"
)

print(
    f"Crime date range: "
    f"{crime['analysis_date'].min().date()} "
    f"to {crime['analysis_date'].max().date()}"
)


# -------------------------------------------------------------------
# Verify that explicit NIBRS "not a crime" records are no longer
# entering the analytical population after the prior notebook's rules.
# -------------------------------------------------------------------

if "nibrs_crime_against_category" in crime.columns:
    crime_against = (
        crime["nibrs_crime_against_category"]
        .astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"[\s\-]+", "_", regex=True)
    )

    remaining_not_a_crime = (
        crime_against.eq("not_a_crime")
    )

    print(
        f"Explicit not_a_crime offenses remaining: "
        f"{crime.loc[remaining_not_a_crime, CRIME_EVENT_ID_COLUMN].nunique():,}"
    )

    if remaining_not_a_crime.any():
        print(
            "WARNING: not_a_crime records still exist in the "
            "analytical population."
        )
    else:
        print(
            "PASS: No explicit not_a_crime records remain "
            "in the analytical population."
        )


# -------------------------------------------------------------------
# Every included offense should have exactly one canonical type.
# -------------------------------------------------------------------

missing_type = ~valid_string_mask(
    crime[CRIME_TYPE_COLUMN]
)

noncanonical_type = (
    ~crime[CRIME_TYPE_COLUMN]
    .isin(CANONICAL_CRIME_TYPES)
)

print(
    f"Missing crime type: "
    f"{crime.loc[missing_type, CRIME_EVENT_ID_COLUMN].nunique():,}"
)

print(
    f"Non-canonical crime type: "
    f"{crime.loc[noncanonical_type, CRIME_EVENT_ID_COLUMN].nunique():,}"
)

POST-CLASSIFICATION CRIME POPULATION
Unique offenses: 138,525
Unique reports:  124,813
Crime date range: 2024-09-09 to 2026-09-11
Explicit not_a_crime offenses remaining: 0
PASS: No explicit not_a_crime records remain in the analytical population.
Missing crime type: 0
Non-canonical crime type: 0


## Current vs previous period methodology

The comparison period is the immediately preceding period containing the exact
same number of calendar days as the selected current period.

Both start and end dates are inclusive.

Example:

Current:
September 1 through September 7 = 7 days

Previous:
August 25 through August 31 = 7 days

There is no gap or overlap between the two periods.

Percentage change is:

(current - previous) / previous × 100

If the previous value is zero, percentage change is undefined rather than
infinite.

Stored history != selectable analysis history. The selectable domain ends at
analysis_end = latest_available_date and begins at
analysis_start = analysis_end - pd.DateOffset(years=1), with no +1 day.
Both endpoints are inclusive, matching native Plotly calendar-year semantics.
A leap-year interval can contain 367 inclusive calendar dates, so a fixed
366-day cap is incorrect.

The user selects only the current period within that domain. The previous
period is automatically derived and may precede analysis_start. Both periods
contain the same number of calendar dates, with no gap and no overlap.
Full stored history remains available for these hidden comparisons.


In [ ]:
def make_equal_periods(
    current_start,
    current_end,
):
    current_start = pd.Timestamp(
        current_start
    ).normalize()

    current_end = pd.Timestamp(
        current_end
    ).normalize()

    if current_start > current_end:
        raise ValueError(
            "current_start must be <= current_end"
        )

    period_days = (
        current_end - current_start
    ).days + 1

    previous_end = (
        current_start
        - pd.Timedelta(days=1)
    )

    previous_start = (
        current_start
        - pd.Timedelta(days=period_days)
    )

    result = {
        "current_start": current_start,
        "current_end": current_end,
        "previous_start": previous_start,
        "previous_end": previous_end,
        "period_days": period_days,
    }

    # Internal correctness checks.
    previous_days = (
        previous_end - previous_start
    ).days + 1

    assert previous_days == period_days

    assert (
        previous_end
        + pd.Timedelta(days=1)
        == current_start
    )

    return result


# -------------------------------------------------------------------
# Basic unit checks.
# -------------------------------------------------------------------

one_day = make_equal_periods(
    "2026-09-06",
    "2026-09-06",
)

assert one_day["previous_start"] == pd.Timestamp(
    "2026-09-05"
)

assert one_day["previous_end"] == pd.Timestamp(
    "2026-09-05"
)


seven_days = make_equal_periods(
    "2026-08-31",
    "2026-09-06",
)

assert seven_days["period_days"] == 7

assert seven_days["previous_start"] == pd.Timestamp(
    "2026-08-24"
)

assert seven_days["previous_end"] == pd.Timestamp(
    "2026-08-30"
)

print("PASS: Equal-period helper checks passed.")

In [ ]:
# A native calendar year can contain 367 inclusive dates.
leap_end = pd.Timestamp("2024-02-29")
leap_start = leap_end - pd.DateOffset(years=1)
leap_periods = make_equal_periods(leap_start, leap_end)

assert leap_start == pd.Timestamp("2023-02-28")
assert leap_periods["period_days"] == 367
assert leap_periods["previous_start"] == pd.Timestamp("2022-02-26")
assert leap_periods["previous_end"] == pd.Timestamp("2023-02-27")
assert (leap_periods["previous_end"] - leap_periods["previous_start"]).days + 1 == 367
assert leap_periods["previous_end"] + pd.Timedelta(days=1) == leap_start
print("PASS: Native leap-year current and previous periods each contain 367 dates.")


In [ ]:
LATEST_CRIME_DATE = (
    crime["analysis_date"]
    .dropna()
    .max()
)

EARLIEST_CRIME_DATE = (
    crime["analysis_date"]
    .dropna()
    .min()
)

CURRENT_END = LATEST_CRIME_DATE

CURRENT_START = CURRENT_END - pd.DateOffset(years=1)

periods = make_equal_periods(
    CURRENT_START,
    CURRENT_END,
)

display(
    pd.DataFrame(
        {
            "period": [
                "Current",
                "Previous",
            ],
            "start": [
                periods["current_start"],
                periods["previous_start"],
            ],
            "end": [
                periods["current_end"],
                periods["previous_end"],
            ],
            "days": [
                periods["period_days"],
                periods["period_days"],
            ],
        }
    )
)

previous_supported = (
    periods["previous_start"]
    >= EARLIEST_CRIME_DATE
)

print(
    f"Previous period fully supported: "
    f"{previous_supported}"
)

In [ ]:
latest_call_date = (
    calls[CALL_TIME_COLUMN]
    .dropna()
    .dt.normalize()
    .max()
)

earliest_call_date = (
    calls[CALL_TIME_COLUMN]
    .dropna()
    .dt.normalize()
    .min()
)


source_coverage = pd.DataFrame(
    {
        "source": [
            "Crime offenses",
            "CAD calls",
        ],
        "earliest_date": [
            EARLIEST_CRIME_DATE,
            earliest_call_date,
        ],
        "latest_date": [
            LATEST_CRIME_DATE,
            latest_call_date,
        ],
    }
)

source_coverage["days_available"] = (
    source_coverage["latest_date"]
    - source_coverage["earliest_date"]
).dt.days + 1

display(source_coverage)


# -------------------------------------------------------------------
# Can a full latest calendar year also have an equal previous period?
# -------------------------------------------------------------------

latest_year_start = (
    LATEST_CRIME_DATE
    - pd.DateOffset(years=1)
)

latest_year_periods = make_equal_periods(
    latest_year_start,
    LATEST_CRIME_DATE,
)

full_year_comparison_supported = (
    latest_year_periods["previous_start"]
    >= EARLIEST_CRIME_DATE
)

print(
    "Latest-year current period:"
)

print(
    latest_year_periods["current_start"].date(),
    "to",
    latest_year_periods["current_end"].date(),
)

print(
    "Required previous period begins:",
    latest_year_periods["previous_start"].date(),
)

print(
    "Full one-year comparison supported:",
    full_year_comparison_supported,
)

In [6]:
SELECTED_CRIME_TYPES = (
    CANONICAL_CRIME_TYPES.copy()
)

SELECTED_SUBCATEGORIES = []

SELECTED_NEIGHBORHOODS = []


def filter_crime_period(
    records,
    start_date,
    end_date,
    crime_types=None,
    subcategories=None,
    neighborhoods=None,
):
    out = records.copy()

    start_date = pd.Timestamp(
        start_date
    ).normalize()

    end_date = pd.Timestamp(
        end_date
    ).normalize()

    dates = pd.to_datetime(
        out[CRIME_TIME_COLUMN],
        errors="coerce",
    ).dt.normalize()

    mask = dates.between(
        start_date,
        end_date,
    )

    if crime_types:
        selected_types = (
            clean_string(
                pd.Series(
                    crime_types,
                    dtype="string",
                )
            )
            .dropna()
            .tolist()
        )

        mask &= (
            clean_string(
                out[CRIME_TYPE_COLUMN]
            )
            .isin(selected_types)
        )

    if subcategories:
        selected_subcategories = (
            clean_string(
                pd.Series(
                    subcategories,
                    dtype="string",
                )
            )
            .dropna()
            .tolist()
        )

        mask &= (
            clean_string(
                out[CRIME_SUBCATEGORY_COLUMN]
            )
            .isin(selected_subcategories)
        )

    if neighborhoods:
        selected_neighborhoods = (
            normalize_neighborhood_name(
                pd.Series(
                    neighborhoods,
                    dtype="string",
                )
            )
            .dropna()
            .tolist()
        )

        record_neighborhoods = (
            normalize_neighborhood_name(
                out["mcpp_neighborhood"]
            )
        )

        mask &= record_neighborhoods.isin(
            selected_neighborhoods
        )

    return out.loc[mask].copy()


current_crime = filter_crime_period(
    crime,
    periods["current_start"],
    periods["current_end"],
    crime_types=SELECTED_CRIME_TYPES,
    subcategories=SELECTED_SUBCATEGORIES,
    neighborhoods=SELECTED_NEIGHBORHOODS,
)

previous_crime = filter_crime_period(
    crime,
    periods["previous_start"],
    periods["previous_end"],
    crime_types=SELECTED_CRIME_TYPES,
    subcategories=SELECTED_SUBCATEGORIES,
    neighborhoods=SELECTED_NEIGHBORHOODS,
)

print(
    f"Current offenses: "
    f"{current_crime[CRIME_EVENT_ID_COLUMN].nunique():,}"
)

print(
    f"Previous offenses: "
    f"{previous_crime[CRIME_EVENT_ID_COLUMN].nunique():,}"
)

Current offenses: 66,773
Previous offenses: 71,247


## Crime counting unit

The headline crime KPI should count unique `offense_id` values.

`report_number` represents a different concept: a single report may contain
multiple separately classified offenses.

Therefore:

**Total Offenses** = unique `offense_id`

**Reports / Incidents** = unique `report_number`

The two measures should never be used interchangeably.

Current-vs-previous crime comparisons use unique offenses.

In [7]:
def safe_pct_change(
    current,
    previous,
):
    if pd.isna(current) or pd.isna(previous):
        return np.nan

    if previous == 0:
        if current == 0:
            return 0.0

        return np.nan

    return (
        (current - previous)
        / previous
        * 100
    )


current_offenses = (
    current_crime[
        CRIME_EVENT_ID_COLUMN
    ]
    .nunique()
)

previous_offenses = (
    previous_crime[
        CRIME_EVENT_ID_COLUMN
    ]
    .nunique()
)

current_reports = (
    current_crime[
        CRIME_REPORT_ID_COLUMN
    ]
    .nunique()
)

previous_reports = (
    previous_crime[
        CRIME_REPORT_ID_COLUMN
    ]
    .nunique()
)


overall_period_comparison = pd.DataFrame(
    {
        "metric": [
            "Unique offenses",
            "Unique reports",
        ],
        "current": [
            current_offenses,
            current_reports,
        ],
        "previous": [
            previous_offenses,
            previous_reports,
        ],
    }
)

overall_period_comparison["raw_change"] = (
    overall_period_comparison["current"]
    - overall_period_comparison["previous"]
)

overall_period_comparison["pct_change"] = (
    overall_period_comparison.apply(
        lambda row: safe_pct_change(
            row["current"],
            row["previous"],
        ),
        axis=1,
    )
)

display(overall_period_comparison)

,metric,current,previous,raw_change,pct_change
0,Unique offenses,66773,71247,-4474,-6.279563
1,Unique reports,60231,64134,-3903,-6.085696


In [8]:
current_by_type = (
    current_crime
    .groupby(
        CRIME_TYPE_COLUMN
    )[CRIME_EVENT_ID_COLUMN]
    .nunique()
    .reindex(
        CANONICAL_CRIME_TYPES,
        fill_value=0,
    )
)

previous_by_type = (
    previous_crime
    .groupby(
        CRIME_TYPE_COLUMN
    )[CRIME_EVENT_ID_COLUMN]
    .nunique()
    .reindex(
        CANONICAL_CRIME_TYPES,
        fill_value=0,
    )
)


crime_type_comparison = pd.DataFrame(
    {
        "crime_type": (
            CANONICAL_CRIME_TYPES
        ),
        "current_offenses": (
            current_by_type.values
        ),
        "previous_offenses": (
            previous_by_type.values
        ),
    }
)

crime_type_comparison["raw_change"] = (
    crime_type_comparison["current_offenses"]
    - crime_type_comparison["previous_offenses"]
)

crime_type_comparison["pct_change"] = (
    crime_type_comparison.apply(
        lambda row: safe_pct_change(
            row["current_offenses"],
            row["previous_offenses"],
        ),
        axis=1,
    )
)

crime_type_comparison[
    "current_share_pct"
] = (
    100
    * crime_type_comparison[
        "current_offenses"
    ]
    / max(
        crime_type_comparison[
            "current_offenses"
        ].sum(),
        1,
    )
)

display(crime_type_comparison)

,crime_type,current_offenses,previous_offenses,raw_change,pct_change,current_share_pct
0,crimes against persons,12213,12156,57,0.468904,18.290327
1,crimes against property,44083,49469,-5386,-10.887627,66.019199
2,crimes against society / other,10477,9622,855,8.885887,15.690474


## Crime-rate methodology

For an arbitrary selected period:

period crime rate per 100,000 =
unique offenses during selected period / population × 100,000

The metric should describe the selected period. It should **not** be silently
annualized.

Because current and previous periods always have equal duration, their rates
can be compared directly.

Before production use, however, the population denominator must have a
documented and defensible source.

The existing neighborhood-population file should not automatically be summed
and treated as Seattle's citywide population without validating its coverage
and source.

In [9]:
population = population.copy()

if "mcpp_neighborhood" in population.columns:
    POPULATION_NEIGHBORHOOD_COLUMN = (
        "mcpp_neighborhood"
    )

elif "dispatch_neighborhood" in population.columns:
    POPULATION_NEIGHBORHOOD_COLUMN = (
        "dispatch_neighborhood"
    )

else:
    raise ValueError(
        "Population data has no recognized "
        "neighborhood column."
    )


population[
    POPULATION_NEIGHBORHOOD_COLUMN
] = normalize_neighborhood_name(
    population[
        POPULATION_NEIGHBORHOOD_COLUMN
    ]
)

population["population"] = pd.to_numeric(
    population["population"],
    errors="coerce",
)


population_by_neighborhood = (
    population
    .groupby(
        POPULATION_NEIGHBORHOOD_COLUMN,
        as_index=False,
    )
    .agg(
        population=(
            "population",
            "sum",
        )
    )
)


print(
    f"Population neighborhoods: "
    f"{len(population_by_neighborhood):,}"
)

print(
    f"Sum of neighborhood population values: "
    f"{population_by_neighborhood['population'].sum():,.0f}"
)

print(
    f"Missing population values: "
    f"{population_by_neighborhood['population'].isna().sum():,}"
)

print(
    f"Zero/non-positive population values: "
    f"{(population_by_neighborhood['population'] <= 0).sum():,}"
)


crime_neighborhoods = set(
    normalize_neighborhood_name(
        crime["mcpp_neighborhood"]
    )
    .dropna()
    .loc[
        lambda s: ~s.isin(
            INVALID_TEXT_VALUES
        )
    ]
    .unique()
)

population_neighborhoods = set(
    population_by_neighborhood[
        POPULATION_NEIGHBORHOOD_COLUMN
    ]
    .dropna()
    .unique()
)

crime_without_population = sorted(
    crime_neighborhoods
    - population_neighborhoods
)

print()
print(
    "Crime neighborhoods without population values:",
    len(crime_without_population),
)

if crime_without_population:
    display(
        pd.DataFrame(
            {
                "crime_neighborhood_without_population": (
                    crime_without_population
                )
            }
        )
    )


# -------------------------------------------------------------------
# Do NOT silently populate this from the neighborhood sum.
# Replace only after choosing a documented citywide source.
# -------------------------------------------------------------------

CITYWIDE_POPULATION = None

POPULATION_SOURCE_CONFIRMED = False

Population neighborhoods: 58
Sum of neighborhood population values: 570,655
Missing population values: 0
Zero/non-positive population values: 1

Crime neighborhoods without population values: 1


,crime_neighborhood_without_population
0,ooj


In [10]:
def rate_per_100k(
    offense_count,
    population_value,
):
    if (
        population_value is None
        or pd.isna(population_value)
        or population_value <= 0
    ):
        return np.nan

    return (
        offense_count
        / population_value
        * 100_000
    )


if (
    CITYWIDE_POPULATION is None
    or not POPULATION_SOURCE_CONFIRMED
):
    print(
        "CITYWIDE CRIME RATE NOT YET FINALIZED."
    )
    print(
        "Choose and document the citywide population "
        "source before exposing this KPI."
    )

else:
    city_rate_comparison = (
        crime_type_comparison.copy()
    )

    city_rate_comparison[
        "current_rate_per_100k"
    ] = (
        city_rate_comparison[
            "current_offenses"
        ]
        .apply(
            lambda count: rate_per_100k(
                count,
                CITYWIDE_POPULATION,
            )
        )
    )

    city_rate_comparison[
        "previous_rate_per_100k"
    ] = (
        city_rate_comparison[
            "previous_offenses"
        ]
        .apply(
            lambda count: rate_per_100k(
                count,
                CITYWIDE_POPULATION,
            )
        )
    )

    display(city_rate_comparison)

CITYWIDE CRIME RATE NOT YET FINALIZED.
Choose and document the citywide population source before exposing this KPI.


## Neighborhood volume rankings

Neighborhood ranking is based on unique offenses, not raw rows and not reports.

Offenses without a valid analytical neighborhood remain in citywide totals but
cannot participate in neighborhood rankings.

Current rank:
rank neighborhoods by current-period unique offense count, descending.

Previous rank:
rank neighborhoods by previous-period unique offense count, descending.

Rank change:
previous rank - current rank

A positive rank change means the neighborhood moved closer to rank #1
(higher crime-volume ranking).

A negative rank change means it moved farther from rank #1.

In [11]:
def prepare_rankable_neighborhoods(
    records,
):
    out = records.copy()

    out["analysis_neighborhood"] = (
        normalize_neighborhood_name(
            out["mcpp_neighborhood"]
        )
    )

    valid_neighborhood = (
        out[
            "analysis_neighborhood"
        ].notna()
        & ~out[
            "analysis_neighborhood"
        ].isin(
            INVALID_TEXT_VALUES
        )
    )

    return (
        out.loc[
            valid_neighborhood
        ].copy()
    )


current_rankable = (
    prepare_rankable_neighborhoods(
        current_crime
    )
)

previous_rankable = (
    prepare_rankable_neighborhoods(
        previous_crime
    )
)


current_neighborhood_counts = (
    current_rankable
    .groupby(
        "analysis_neighborhood"
    )[CRIME_EVENT_ID_COLUMN]
    .nunique()
    .rename("current_offenses")
)

previous_neighborhood_counts = (
    previous_rankable
    .groupby(
        "analysis_neighborhood"
    )[CRIME_EVENT_ID_COLUMN]
    .nunique()
    .rename("previous_offenses")
)


neighborhood_volume = (
    pd.concat(
        [
            current_neighborhood_counts,
            previous_neighborhood_counts,
        ],
        axis=1,
    )
    .fillna(0)
    .reset_index()
)

neighborhood_volume[
    "current_offenses"
] = (
    neighborhood_volume[
        "current_offenses"
    ].astype(int)
)

neighborhood_volume[
    "previous_offenses"
] = (
    neighborhood_volume[
        "previous_offenses"
    ].astype(int)
)


neighborhood_volume["current_rank"] = (
    neighborhood_volume[
        "current_offenses"
    ]
    .rank(
        method="min",
        ascending=False,
    )
)

neighborhood_volume["previous_rank"] = (
    neighborhood_volume[
        "previous_offenses"
    ]
    .rank(
        method="min",
        ascending=False,
    )
)

neighborhood_volume["rank_change"] = (
    neighborhood_volume["previous_rank"]
    - neighborhood_volume["current_rank"]
)

neighborhood_volume["raw_change"] = (
    neighborhood_volume["current_offenses"]
    - neighborhood_volume["previous_offenses"]
)

neighborhood_volume["pct_change"] = (
    neighborhood_volume.apply(
        lambda row: safe_pct_change(
            row["current_offenses"],
            row["previous_offenses"],
        ),
        axis=1,
    )
)


top_neighborhoods = (
    neighborhood_volume
    .sort_values(
        [
            "current_rank",
            "analysis_neighborhood",
        ]
    )
    .head(10)
)

display(top_neighborhoods)


current_unrankable = (
    current_crime.loc[
        ~current_crime.index.isin(
            current_rankable.index
        ),
        CRIME_EVENT_ID_COLUMN,
    ]
    .nunique()
)

print(
    f"Current-period offenses excluded from "
    f"neighborhood rankings only: "
    f"{current_unrankable:,}"
)

,analysis_neighborhood,current_offenses,previous_offenses,current_rank,previous_rank,rank_change,raw_change,pct_change
7,capitol hill,5346,5674,1.0,1.0,0.0,-328,-5.780754
46,queen anne,3998,3676,2.0,4.0,2.0,322,8.759521
52,slu/cascade,3511,3752,3.0,3.0,0.0,-241,-6.423241
14,downtown commercial,3462,3827,4.0,2.0,-2.0,-365,-9.537497
41,northgate,3166,3554,5.0,5.0,0.0,-388,-10.917276
18,first hill,2855,2787,6.0,7.0,1.0,68,2.439900
57,university,2766,2969,7.0,6.0,-1.0,-203,-6.837319
3,ballard south,2702,2661,8.0,8.0,0.0,41,1.540774
9,chinatown/international district,2344,2283,9.0,10.0,1.0,61,2.671923
49,roosevelt/ravenna,2309,2607,10.0,9.0,-1.0,-298,-11.430763


Current-period offenses excluded from neighborhood rankings only: 763


## Map coverage

The map must not imply that all analytically counted offenses are visible as
points.

For the current selected analytical population:

unmappable percentage =
1 - (unique mappable offense IDs / total unique offense IDs)

This calculation must happen **after** the active crime filters are applied.

Unmappable offenses remain in citywide totals, time-series values, KPIs, and
period comparisons.

Only geographic visualization requires mappability.

In [12]:
mappable_crime_ids = set(
    clean_string(
        mappable_crime[
            CRIME_EVENT_ID_COLUMN
        ]
    )
    .dropna()
    .tolist()
)


def calculate_map_coverage(
    selected_records,
):
    selected_ids = set(
        clean_string(
            selected_records[
                CRIME_EVENT_ID_COLUMN
            ]
        )
        .dropna()
        .tolist()
    )

    total = len(selected_ids)

    if total == 0:
        return {
            "total_offenses": 0,
            "mappable_offenses": 0,
            "unmappable_offenses": 0,
            "mappable_pct": np.nan,
            "unmappable_pct": np.nan,
        }

    mappable = len(
        selected_ids
        & mappable_crime_ids
    )

    unmappable = (
        total - mappable
    )

    return {
        "total_offenses": total,
        "mappable_offenses": mappable,
        "unmappable_offenses": unmappable,
        "mappable_pct": (
            100 * mappable / total
        ),
        "unmappable_pct": (
            100 * unmappable / total
        ),
    }


current_map_coverage = (
    calculate_map_coverage(
        current_crime
    )
)

previous_map_coverage = (
    calculate_map_coverage(
        previous_crime
    )
)

display(
    pd.DataFrame(
        [
            {
                "period": "Current",
                **current_map_coverage,
            },
            {
                "period": "Previous",
                **previous_map_coverage,
            },
        ]
    )
)


if not pd.isna(
    current_map_coverage[
        "unmappable_pct"
    ]
):
    MAP_ANNOTATION_TEXT = (
        f"{current_map_coverage['unmappable_pct']:.1f}% "
        "of offenses in this selection cannot be mapped. "
        "Citywide totals include these offenses."
    )

    print()
    print("Candidate map annotation:")
    print(MAP_ANNOTATION_TEXT)

,period,total_offenses,mappable_offenses,unmappable_offenses,mappable_pct,unmappable_pct
0,Current,66773,55920,10853,83.746424,16.253576
1,Previous,71247,61675,9572,86.565048,13.434952



Candidate map annotation:
16.3% of offenses in this selection cannot be mapped. Citywide totals include these offenses.


## Shooting KPI methodology

Before counting "shootings", inspect the actual values in
`shooting_type_group`.

We also need to determine the counting unit.

Because multiple offenses may exist on one report, a shooting KPI could differ
substantially depending on whether it counts:

- unique offenses, or
- unique reports / incidents.

No shooting KPI should be implemented until both the qualifying field values
and the counting unit have been reviewed.

In [13]:
SHOOTING_COLUMN = (
    "shooting_type_group"
)

if SHOOTING_COLUMN not in crime.columns:
    raise ValueError(
        f"Missing required field: "
        f"{SHOOTING_COLUMN}"
    )


shooting_audit = crime.copy()

shooting_audit[
    "shooting_value"
] = (
    clean_string(
        shooting_audit[
            SHOOTING_COLUMN
        ]
    )
    .fillna("<missing>")
)

shooting_value_summary = (
    shooting_audit
    .groupby(
        "shooting_value",
        dropna=False,
    )
    .agg(
        rows=(
            CRIME_EVENT_ID_COLUMN,
            "size",
        ),
        unique_offenses=(
            CRIME_EVENT_ID_COLUMN,
            "nunique",
        ),
        unique_reports=(
            CRIME_REPORT_ID_COLUMN,
            "nunique",
        ),
    )
    .reset_index()
    .sort_values(
        "unique_offenses",
        ascending=False,
    )
)

shooting_value_summary[
    "offenses_per_report"
] = (
    shooting_value_summary[
        "unique_offenses"
    ]
    / shooting_value_summary[
        "unique_reports"
    ].replace(
        0,
        np.nan,
    )
)

display(shooting_value_summary)


shooting_crosswalk = (
    shooting_audit
    .groupby(
        [
            "shooting_value",
            CRIME_TYPE_COLUMN,
            CRIME_SUBCATEGORY_COLUMN,
        ],
        dropna=False,
    )
    .agg(
        unique_offenses=(
            CRIME_EVENT_ID_COLUMN,
            "nunique",
        ),
        unique_reports=(
            CRIME_REPORT_ID_COLUMN,
            "nunique",
        ),
    )
    .reset_index()
    .sort_values(
        "unique_offenses",
        ascending=False,
    )
)

display(shooting_crosswalk)

,shooting_value,rows,unique_offenses,unique_reports,offenses_per_report
0,-,137043,137043,123727,1.107624
3,shots fired (eyewitness/casings/property damage),1160,1160,852,1.361502
2,shooting (non-fatal injury),260,260,193,1.347150
1,shooting (fatal injury),62,62,42,1.476190


,shooting_value,event_importance_bin,offense_sub_category,unique_offenses,unique_reports
11,-,crimes against property,larceny-theft,46551,46367
9,-,crimes against property,burglary,15381,15381
13,-,crimes against property,"property offenses (includes stolen, destruction)",13381,13367
1,-,crimes against persons,assault offenses,13193,13086
12,-,crimes against property,motor vehicle theft,11298,11298
10,-,crimes against property,extortion/fraud/forgery/bribery (includes bad checks),6693,6357
0,-,crimes against persons,aggravated assault,5969,5969
14,-,crimes against society / other,all other,5154,5100
20,-,crimes against society / other,narcotic violations (includes drug equip.),3780,3371
25,-,crimes against society / other,trespass,3078,3077


In [14]:
# -------------------------------------------------------------------
# Fill these only after reviewing Cell 19.
#
# Example only:
#
# SHOOTING_VALUE_DECISIONS = {
#     "some confirmed shooting value": True,
#     "some confirmed non-shooting value": False,
# }
# -------------------------------------------------------------------

SHOOTING_VALUE_DECISIONS = {
}


# Does a missing shooting_type_group mean "not a shooting"?
#
# Set to True or False only after confirming the field semantics.

MISSING_SHOOTING_MEANS_NO = None


# Final count unit should be one of:
#
# "offense"
# "report"

SHOOTING_COUNT_UNIT = None


observed_shooting_values = sorted(
    set(
        shooting_audit[
            "shooting_value"
        ].unique()
    )
    - {"<missing>"}
)

unresolved_shooting_values = [
    value
    for value in observed_shooting_values
    if value
    not in SHOOTING_VALUE_DECISIONS
]

print(
    "Unresolved shooting values:",
    unresolved_shooting_values,
)

print(
    "Missing-value semantics resolved:",
    MISSING_SHOOTING_MEANS_NO
    is not None,
)

print(
    "Shooting count unit:",
    SHOOTING_COUNT_UNIT,
)

Unresolved shooting values: ['-', 'shooting (fatal injury)', 'shooting (non-fatal injury)', 'shots fired (eyewitness/casings/property damage)']
Missing-value semantics resolved: False
Shooting count unit: None


## Qualified CAD response time

Response time is an event-level CAD metric.

Candidate event definition:

queued time = earliest queue timestamp for a CAD event

first arrival time = earliest recorded arrival timestamp for that CAD event

response time =
first arrival time - queued time

The current production pipeline accepts response times from 0 through 24 hours.

This notebook first reproduces that existing implementation, then evaluates:

- missing arrival times
- negative response times
- extreme response times
- priority composition
- alternative maximum-response thresholds
- the effect of limiting the KPI to dispatch-response priorities

Coordinates are **not** required for response-time inclusion.

In [15]:
response_source = calls[
    calls[
        CALL_EVENT_ID_COLUMN
    ].notna()
].copy()

response_source = (
    response_source
    .sort_values(
        CALL_TIME_COLUMN
    )
)


response_agg = {
    "queued_time": (
        CALL_TIME_COLUMN,
        "min",
    ),
    "first_arrival_time": (
        CALL_ARRIVAL_COLUMN,
        "min",
    ),
    "priority": (
        "priority",
        "first",
    ),
    "dispatch_neighborhood": (
        "dispatch_neighborhood",
        "first",
    ),
    "event_group": (
        "event_group",
        "first",
    ),
}

if (
    "event_importance_bin"
    in response_source.columns
):
    response_agg[
        "event_importance_bin"
    ] = (
        "event_importance_bin",
        "first",
    )


response_events = (
    response_source
    .groupby(
        CALL_EVENT_ID_COLUMN,
        as_index=False,
    )
    .agg(
        **response_agg
    )
)


response_events[
    "response_time_minutes"
] = (
    response_events[
        "first_arrival_time"
    ]
    - response_events[
        "queued_time"
    ]
).dt.total_seconds() / 60


response_events[
    "queued_date"
] = (
    pd.to_datetime(
        response_events[
            "queued_time"
        ],
        errors="coerce",
    )
    .dt.normalize()
)


response_events[
    "priority"
] = pd.to_numeric(
    response_events[
        "priority"
    ],
    errors="coerce",
)


response_events[
    "dispatch_neighborhood"
] = clean_string(
    response_events[
        "dispatch_neighborhood"
    ]
)


CURRENT_PRODUCTION_MAX_MINUTES = (
    24 * 60
)


response_events[
    "current_production_qualified"
] = (
    response_events[
        "response_time_minutes"
    ].notna()
    & response_events[
        "response_time_minutes"
    ].between(
        0,
        CURRENT_PRODUCTION_MAX_MINUTES,
    )
)


# -------------------------------------------------------------------
# Verify that our reconstruction agrees with the current production
# response-analysis population.
# -------------------------------------------------------------------

existing_response_ids = set(
    clean_string(
        existing_response_analysis[
            CALL_EVENT_ID_COLUMN
        ]
    )
    .dropna()
    .tolist()
)

reconstructed_qualified_ids = set(
    response_events.loc[
        response_events[
            "current_production_qualified"
        ],
        CALL_EVENT_ID_COLUMN,
    ]
    .dropna()
    .tolist()
)


print(
    f"Existing response-analysis events: "
    f"{len(existing_response_ids):,}"
)

print(
    f"Reconstructed current-rule events: "
    f"{len(reconstructed_qualified_ids):,}"
)

print(
    "Exact ID match:",
    existing_response_ids
    == reconstructed_qualified_ids,
)

Existing response-analysis events: 607,825
Reconstructed current-rule events: 607,825
Exact ID match: True


In [16]:
total_cad_events = len(
    response_events
)

has_queue = (
    response_events[
        "queued_time"
    ].notna()
)

has_arrival = (
    response_events[
        "first_arrival_time"
    ].notna()
)

has_calculated_response = (
    response_events[
        "response_time_minutes"
    ].notna()
)

nonnegative_response = (
    has_calculated_response
    & (
        response_events[
            "response_time_minutes"
        ]
        >= 0
    )
)

within_24_hours = (
    nonnegative_response
    & (
        response_events[
            "response_time_minutes"
        ]
        <= CURRENT_PRODUCTION_MAX_MINUTES
    )
)


response_funnel = pd.DataFrame(
    {
        "stage": [
            "Unique CAD events",
            "Has queued time",
            "Has arrival time",
            "Response can be calculated",
            "Response >= 0 minutes",
            "Response <= 24 hours",
        ],
        "events": [
            total_cad_events,
            int(has_queue.sum()),
            int(has_arrival.sum()),
            int(
                has_calculated_response.sum()
            ),
            int(
                nonnegative_response.sum()
            ),
            int(
                within_24_hours.sum()
            ),
        ],
    }
)

response_funnel[
    "share_of_all_events_pct"
] = (
    100
    * response_funnel["events"]
    / total_cad_events
)

display(response_funnel)

,stage,events,share_of_all_events_pct
0,Unique CAD events,679355,100.000000
1,Has queued time,679355,100.000000
2,Has arrival time,608741,89.605729
3,Response can be calculated,608741,89.605729
4,Response >= 0 minutes,608719,89.602491
5,Response <= 24 hours,607825,89.470895


In [17]:
PRIORITY_LABELS = {
    1: (
        "Incidents posing an imminent "
        "threat to life"
    ),
    2: (
        "Urgent (non-life threatening)"
    ),
    3: (
        "Non-emergency (Routine)"
    ),
    4: (
        "Administrative calls, cold "
        "incidents (Low-Level)"
    ),
    5: (
        "Alternative/Telephone reporting"
    ),
    7: (
        "Officer-Initiated Activity"
    ),
    9: (
        "Lowest Urgency / Information only"
    ),
}


priority_rows = []

for priority, group in (
    response_events
    .groupby(
        "priority",
        dropna=False,
    )
):
    qualified = group[
        group[
            "response_time_minutes"
        ].between(
            0,
            CURRENT_PRODUCTION_MAX_MINUTES,
        )
    ]

    negative_count = int(
        (
            group[
                "response_time_minutes"
            ]
            < 0
        ).sum()
    )

    over_24h_count = int(
        (
            group[
                "response_time_minutes"
            ]
            > CURRENT_PRODUCTION_MAX_MINUTES
        ).sum()
    )

    priority_key = (
        int(priority)
        if pd.notna(priority)
        else None
    )

    priority_rows.append(
        {
            "priority": priority,
            "label": (
                PRIORITY_LABELS.get(
                    priority_key,
                    "Unknown / missing",
                )
            ),
            "cad_events": len(group),
            "with_arrival": int(
                group[
                    "first_arrival_time"
                ].notna().sum()
            ),
            "qualified_24h": (
                len(qualified)
            ),
            "negative_response": (
                negative_count
            ),
            "over_24h": (
                over_24h_count
            ),
            "median_minutes": (
                qualified[
                    "response_time_minutes"
                ].median()
            ),
            "p90_minutes": (
                qualified[
                    "response_time_minutes"
                ].quantile(0.90)
            ),
            "p95_minutes": (
                qualified[
                    "response_time_minutes"
                ].quantile(0.95)
            ),
            "max_minutes": (
                qualified[
                    "response_time_minutes"
                ].max()
            ),
        }
    )


priority_response_summary = (
    pd.DataFrame(
        priority_rows
    )
    .sort_values(
        "priority",
        na_position="last",
    )
)

display(priority_response_summary)

,priority,label,cad_events,with_arrival,qualified_24h,negative_response,over_24h,median_minutes,p90_minutes,p95_minutes,max_minutes
0,1,Incidents posing an imminent threat to life,66081,61093,61086,7,0,6.916667,16.850000,22.233333,1333.533333
1,2,Urgent (non-life threatening),216892,200612,200564,7,41,24.833333,176.700000,268.480833,1376.483333
2,3,Non-emergency (Routine),173414,157324,156744,5,575,35.383333,355.811667,536.809167,1439.950000
3,4,"Administrative calls, cold incidents (Low-Level)",63522,31215,30959,2,254,48.583333,437.473333,626.786667,1439.483333
4,5,Alternative/Telephone reporting,47350,46418,46394,0,24,32.541667,356.550000,467.222500,1433.750000
5,7,Officer-Initiated Activity,100026,100016,100015,1,0,0.000000,0.000000,0.016667,306.200000
6,9,Lowest Urgency / Information only,12070,12063,12063,0,0,0.000000,0.000000,0.016667,481.700000


In [18]:
CANDIDATE_QUALIFIED_PRIORITIES = [
    1,
    2,
    3,
]


candidate_priority_events = (
    response_events[
        response_events[
            "priority"
        ].isin(
            CANDIDATE_QUALIFIED_PRIORITIES
        )
        & response_events[
            "response_time_minutes"
        ].notna()
        & (
            response_events[
                "response_time_minutes"
            ]
            >= 0
        )
    ]
    .copy()
)


print(
    f"Nonnegative response observations "
    f"for priorities 1-3: "
    f"{len(candidate_priority_events):,}"
)

Nonnegative response observations for priorities 1-3: 419,010


In [19]:
RESPONSE_CAPS_MINUTES = [
    60,
    120,
    180,
    240,
    480,
    1440,
]


base_candidate_count = len(
    candidate_priority_events
)

cap_rows = []

for cap in RESPONSE_CAPS_MINUTES:
    retained = (
        candidate_priority_events[
            candidate_priority_events[
                "response_time_minutes"
            ]
            <= cap
        ]
    )

    cap_rows.append(
        {
            "max_response_minutes": cap,
            "qualified_events": len(
                retained
            ),
            "retained_pct": (
                100
                * len(retained)
                / base_candidate_count
                if base_candidate_count
                else np.nan
            ),
            "median_minutes": (
                retained[
                    "response_time_minutes"
                ].median()
            ),
            "p90_minutes": (
                retained[
                    "response_time_minutes"
                ].quantile(0.90)
            ),
            "p95_minutes": (
                retained[
                    "response_time_minutes"
                ].quantile(0.95)
            ),
            "max_observed_minutes": (
                retained[
                    "response_time_minutes"
                ].max()
            ),
        }
    )


response_cap_sensitivity = (
    pd.DataFrame(
        cap_rows
    )
)

display(response_cap_sensitivity)

,max_response_minutes,qualified_events,retained_pct,median_minutes,p90_minutes,p95_minutes,max_observed_minutes
0,60,290542,69.340111,8.850000,39.450000,48.750000,60.00
1,120,339381,80.995919,11.766667,73.816667,93.733333,120.00
2,180,365074,87.127754,13.633333,102.483333,134.116667,180.00
3,240,380442,90.795446,14.933333,125.450000,169.483333,240.00
4,480,406219,96.947328,17.450000,181.603333,269.916667,480.00
5,1440,418394,99.852987,18.833333,222.361667,359.789167,1439.95


In [20]:
# -------------------------------------------------------------------
# Candidate v1.1 methodology.
#
# Revisit MAX_QUALIFIED_RESPONSE_MINUTES after reviewing Cell 26.
# -------------------------------------------------------------------

QUALIFIED_RESPONSE_PRIORITIES = [
    1,
    2,
    3,
]

MAX_QUALIFIED_RESPONSE_MINUTES = (
    1440
)

MIN_NEIGHBORHOOD_RESPONSE_EVENTS = (
    30
)


def qualified_response_events(
    response_data,
    start_date,
    end_date,
    priorities=QUALIFIED_RESPONSE_PRIORITIES,
    max_minutes=MAX_QUALIFIED_RESPONSE_MINUTES,
):
    start_date = pd.Timestamp(
        start_date
    ).normalize()

    end_date = pd.Timestamp(
        end_date
    ).normalize()

    dates = pd.to_datetime(
        response_data[
            "queued_time"
        ],
        errors="coerce",
    ).dt.normalize()

    mask = (
        dates.between(
            start_date,
            end_date,
        )
        & response_data[
            "priority"
        ].isin(priorities)
        & response_data[
            "response_time_minutes"
        ].notna()
        & response_data[
            "response_time_minutes"
        ].between(
            0,
            max_minutes,
        )
    )

    return (
        response_data.loc[
            mask
        ].copy()
    )


current_response = (
    qualified_response_events(
        response_events,
        periods["current_start"],
        periods["current_end"],
    )
)

previous_response = (
    qualified_response_events(
        response_events,
        periods["previous_start"],
        periods["previous_end"],
    )
)


print(
    f"Current qualified CAD events: "
    f"{len(current_response):,}"
)

print(
    f"Previous qualified CAD events: "
    f"{len(previous_response):,}"
)

Current qualified CAD events: 208,923
Previous qualified CAD events: 207,021


In [21]:
def summarize_response_period(
    response_data,
):
    if response_data.empty:
        return {
            "qualified_cad_events": 0,
            "median_response_minutes": np.nan,
            "mean_response_minutes": np.nan,
            "p90_response_minutes": np.nan,
            "p95_response_minutes": np.nan,
        }

    response_minutes = (
        response_data[
            "response_time_minutes"
        ]
    )

    return {
        "qualified_cad_events": (
            len(response_data)
        ),
        "median_response_minutes": (
            response_minutes.median()
        ),
        "mean_response_minutes": (
            response_minutes.mean()
        ),
        "p90_response_minutes": (
            response_minutes.quantile(
                0.90
            )
        ),
        "p95_response_minutes": (
            response_minutes.quantile(
                0.95
            )
        ),
    }


current_response_summary = (
    summarize_response_period(
        current_response
    )
)

previous_response_summary = (
    summarize_response_period(
        previous_response
    )
)


response_metric_rows = []

for metric in (
    current_response_summary.keys()
):
    current_value = (
        current_response_summary[
            metric
        ]
    )

    previous_value = (
        previous_response_summary[
            metric
        ]
    )

    response_metric_rows.append(
        {
            "metric": metric,
            "current": current_value,
            "previous": previous_value,
            "raw_change": (
                current_value
                - previous_value
                if (
                    pd.notna(current_value)
                    and pd.notna(
                        previous_value
                    )
                )
                else np.nan
            ),
            "pct_change": (
                safe_pct_change(
                    current_value,
                    previous_value,
                )
            ),
        }
    )


response_period_comparison = (
    pd.DataFrame(
        response_metric_rows
    )
)

display(response_period_comparison)

,metric,current,previous,raw_change,pct_change
0,qualified_cad_events,208923.000000,207021.000000,1902.000000,0.918747
1,median_response_minutes,17.833333,19.950000,-2.116667,-10.609858
2,mean_response_minutes,72.929867,82.657743,-9.727876,-11.768862
3,p90_response_minutes,207.033333,236.050000,-29.016667,-12.292593
4,p95_response_minutes,338.796667,378.783333,-39.986667,-10.556607


## Neighborhood response-time ranking

Neighborhood response-time rankings use qualified CAD events only.

They do not require usable coordinates.

A valid dispatch neighborhood is required because the metric is inherently
geographic.

Neighborhoods with too few qualified events should not be ranked because a
median based on a tiny sample is unstable.

Candidate threshold:

30 qualified CAD events during the selected period.

Worst-response rank #1 means the highest median response time.

For rank change:

previous worst-response rank - current worst-response rank

A positive value means the neighborhood moved upward toward the worse end of
the ranking.

In [22]:
def prepare_response_neighborhood_stats(
    response_data,
    min_events=(
        MIN_NEIGHBORHOOD_RESPONSE_EVENTS
    ),
):
    out = response_data.copy()

    out[
        "analysis_neighborhood"
    ] = normalize_neighborhood_name(
        out[
            "dispatch_neighborhood"
        ]
    )

    valid_neighborhood = (
        out[
            "analysis_neighborhood"
        ].notna()
        & ~out[
            "analysis_neighborhood"
        ].isin(
            INVALID_TEXT_VALUES
        )
    )

    out = out.loc[
        valid_neighborhood
    ].copy()

    summary = (
        out
        .groupby(
            "analysis_neighborhood",
            as_index=False,
        )
        .agg(
            qualified_events=(
                CALL_EVENT_ID_COLUMN,
                "nunique",
            ),
            median_response_minutes=(
                "response_time_minutes",
                "median",
            ),
            mean_response_minutes=(
                "response_time_minutes",
                "mean",
            ),
        )
    )

    summary = summary[
        summary[
            "qualified_events"
        ]
        >= min_events
    ].copy()

    summary["worst_response_rank"] = (
        summary[
            "median_response_minutes"
        ]
        .rank(
            method="min",
            ascending=False,
        )
    )

    return summary


current_response_neighborhoods = (
    prepare_response_neighborhood_stats(
        current_response
    )
)

previous_response_neighborhoods = (
    prepare_response_neighborhood_stats(
        previous_response
    )
)


current_response_neighborhoods = (
    current_response_neighborhoods
    .rename(
        columns={
            "qualified_events": (
                "current_qualified_events"
            ),
            "median_response_minutes": (
                "current_median_minutes"
            ),
            "mean_response_minutes": (
                "current_mean_minutes"
            ),
            "worst_response_rank": (
                "current_worst_rank"
            ),
        }
    )
)

previous_response_neighborhoods = (
    previous_response_neighborhoods
    .rename(
        columns={
            "qualified_events": (
                "previous_qualified_events"
            ),
            "median_response_minutes": (
                "previous_median_minutes"
            ),
            "mean_response_minutes": (
                "previous_mean_minutes"
            ),
            "worst_response_rank": (
                "previous_worst_rank"
            ),
        }
    )
)


response_neighborhood_comparison = (
    current_response_neighborhoods
    .merge(
        previous_response_neighborhoods,
        on="analysis_neighborhood",
        how="outer",
    )
)


response_neighborhood_comparison[
    "median_change_minutes"
] = (
    response_neighborhood_comparison[
        "current_median_minutes"
    ]
    - response_neighborhood_comparison[
        "previous_median_minutes"
    ]
)

response_neighborhood_comparison[
    "median_pct_change"
] = (
    response_neighborhood_comparison
    .apply(
        lambda row: safe_pct_change(
            row[
                "current_median_minutes"
            ],
            row[
                "previous_median_minutes"
            ],
        ),
        axis=1,
    )
)

response_neighborhood_comparison[
    "rank_change_toward_worse"
] = (
    response_neighborhood_comparison[
        "previous_worst_rank"
    ]
    - response_neighborhood_comparison[
        "current_worst_rank"
    ]
)


worst_current_response = (
    response_neighborhood_comparison
    .sort_values(
        "current_worst_rank",
        na_position="last",
    )
    .head(10)
)

display(worst_current_response)

,analysis_neighborhood,current_qualified_events,current_median_minutes,current_mean_minutes,current_worst_rank,previous_qualified_events,previous_median_minutes,previous_mean_minutes,previous_worst_rank,median_change_minutes,median_pct_change,rank_change_toward_worse
16,eastlake - west,698,39.033333,132.143983,1.0,690,54.333333,144.653382,1.0,-15.300000,-28.159509,0.0
31,magnolia,2276,33.008333,109.773967,2.0,2190,31.216667,105.344635,17.0,1.791667,5.739455,15.0
57,wallingford,2555,32.033333,100.716556,3.0,2393,51.650000,116.262864,2.0,-19.616667,-37.979994,-1.0
30,madrona/leschi,1856,31.416667,86.224021,4.0,1874,31.900000,80.470100,16.0,-0.483333,-1.515152,12.0
19,fremont,2785,31.066667,103.996780,5.0,2445,46.666667,124.311772,3.0,-15.600000,-33.428571,-2.0
33,miller park,1354,29.716667,80.384798,6.0,1289,37.233333,97.572227,11.0,-7.516667,-20.188004,5.0
3,ballard south,6922,29.716667,99.526941,7.0,6912,43.075000,111.830331,6.0,-13.358333,-31.011801,-1.0
2,ballard north,3089,29.350000,100.967870,8.0,2853,44.366667,124.792528,4.0,-15.016667,-33.846732,-4.0
48,roosevelt/ravenna,5560,28.775000,100.762665,9.0,5177,44.150000,126.016528,5.0,-15.375000,-34.824462,-4.0
29,madison park,428,28.333333,74.798481,10.0,479,37.900000,96.456924,10.0,-9.566667,-25.241865,0.0


In [23]:
crime_neighborhood_set = set(
    normalize_neighborhood_name(
        crime[
            "mcpp_neighborhood"
        ]
    )
    .dropna()
    .loc[
        lambda s: ~s.isin(
            INVALID_TEXT_VALUES
        )
    ]
    .unique()
)


call_neighborhood_set = set(
    normalize_neighborhood_name(
        calls[
            "dispatch_neighborhood"
        ]
    )
    .dropna()
    .loc[
        lambda s: ~s.isin(
            INVALID_TEXT_VALUES
        )
    ]
    .unique()
)


shared_neighborhoods = (
    crime_neighborhood_set
    & call_neighborhood_set
)

crime_only_neighborhoods = sorted(
    crime_neighborhood_set
    - call_neighborhood_set
)

calls_only_neighborhoods = sorted(
    call_neighborhood_set
    - crime_neighborhood_set
)


print(
    f"Crime neighborhoods: "
    f"{len(crime_neighborhood_set):,}"
)

print(
    f"CAD neighborhoods: "
    f"{len(call_neighborhood_set):,}"
)

print(
    f"Exact normalized overlap: "
    f"{len(shared_neighborhoods):,}"
)

print()

print("Crime-only neighborhood names:")
display(
    pd.DataFrame(
        {
            "crime_only": (
                crime_only_neighborhoods
            )
        }
    )
)

print("CAD-only neighborhood names:")
display(
    pd.DataFrame(
        {
            "calls_only": (
                calls_only_neighborhoods
            )
        }
    )
)

Crime neighborhoods: 59
CAD neighborhoods: 58
Exact normalized overlap: 58

Crime-only neighborhood names:


,crime_only
0,ooj


CAD-only neighborhood names:


,calls_only


In [24]:
control_compatibility = pd.DataFrame(
    [
        {
            "metric": "Total offenses",
            "date": "Yes",
            "crime_type": "Yes",
            "subcategory": "Yes",
            "neighborhood": "Yes",
            "priority": "No",
        },
        {
            "metric": "Crime type totals / rates",
            "date": "Yes",
            "crime_type": "Yes",
            "subcategory": "Yes",
            "neighborhood": "Yes",
            "priority": "No",
        },
        {
            "metric": "Neighborhood crime ranking",
            "date": "Yes",
            "crime_type": "Yes",
            "subcategory": "Yes",
            "neighborhood": "Yes",
            "priority": "No",
        },
        {
            "metric": "Map coverage",
            "date": "Yes",
            "crime_type": "Yes",
            "subcategory": "Yes",
            "neighborhood": "Yes",
            "priority": "No",
        },
        {
            "metric": "Shootings",
            "date": "Yes",
            "crime_type": "Yes",
            "subcategory": "Yes",
            "neighborhood": "Yes",
            "priority": "No",
        },
        {
            "metric": "CAD response time",
            "date": "Yes",
            "crime_type": (
                "No validated offense/CAD link"
            ),
            "subcategory": (
                "No validated offense/CAD link"
            ),
            "neighborhood": (
                "Yes after geography validation"
            ),
            "priority": "Yes",
        },
    ]
)

display(control_compatibility)

,metric,date,crime_type,subcategory,neighborhood,priority
0,Total offenses,Yes,Yes,Yes,Yes,No
1,Crime type totals / rates,Yes,Yes,Yes,Yes,No
2,Neighborhood crime ranking,Yes,Yes,Yes,Yes,No
3,Map coverage,Yes,Yes,Yes,Yes,No
4,Shootings,Yes,Yes,Yes,Yes,No
5,CAD response time,Yes,No validated offense/CAD link,No validated offense/CAD link,Yes after geography validation,Yes


In [25]:
canonical_type_ready = (
    len(unexpected_types) == 0
)

comparison_history_ready = (
    periods["previous_start"]
    >= EARLIEST_CRIME_DATE
)

population_ready = (
    CITYWIDE_POPULATION is not None
    and POPULATION_SOURCE_CONFIRMED
)

shooting_ready = (
    len(
        unresolved_shooting_values
    ) == 0
    and (
        MISSING_SHOOTING_MEANS_NO
        is not None
    )
    and SHOOTING_COUNT_UNIT
    in {
        "offense",
        "report",
    }
)

response_reproduction_ready = (
    existing_response_ids
    == reconstructed_qualified_ids
)


methodology_status = pd.DataFrame(
    {
        "methodology": [
            "Canonical crime bins",
            "Equal previous-period history",
            "Citywide population denominator",
            "Shooting definition",
            "Current response pipeline reproduced",
            "Dynamic map-coverage metric",
        ],
        "ready": [
            canonical_type_ready,
            comparison_history_ready,
            population_ready,
            shooting_ready,
            response_reproduction_ready,
            True,
        ],
        "note": [
            (
                "All included offenses should use "
                "exactly one of the three v1.1 bins."
            ),
            (
                "Previous period must exist for "
                "the selected current period."
            ),
            (
                "Requires documented population "
                "source before rates go live."
            ),
            (
                "Requires field-value semantics "
                "and offense-vs-report decision."
            ),
            (
                "Notebook event IDs should exactly "
                "match current production logic."
            ),
            (
                "Use filtered analytical population; "
                "do not exclude unmappable offenses "
                "from other metrics."
            ),
        ],
    }
)

display(methodology_status)


print()
print("CURRENT METHODOLOGY CANDIDATES")
print("=" * 60)

print(
    "Crime counting unit: unique offense_id"
)

print(
    "Previous comparison: immediately preceding "
    "equal-length period"
)

print(
    "Neighborhood crime ranking: unique offense volume"
)

print(
    "Response-time counting unit: unique CAD event"
)

print(
    "Candidate response priorities:",
    QUALIFIED_RESPONSE_PRIORITIES,
)

print(
    "Current candidate max response:",
    MAX_QUALIFIED_RESPONSE_MINUTES,
    "minutes",
)

print(
    "Candidate minimum neighborhood response sample:",
    MIN_NEIGHBORHOOD_RESPONSE_EVENTS,
)

print()

if not pd.isna(
    current_map_coverage[
        "unmappable_pct"
    ]
):
    print(
        "Current example unmappable share:",
        f"{current_map_coverage['unmappable_pct']:.2f}%",
    )

print()
print(
    "Report-before-offense timestamp anomalies from "
    "the prior QA do not affect ordinary offense-count "
    "metrics; they remain relevant if reporting-lag "
    "analysis is introduced later."
)

,methodology,ready,note
0,Canonical crime bins,True,All included offenses should use exactly one of the three v1.1 bins.
1,Equal previous-period history,True,Previous period must exist for the selected current period.
2,Citywide population denominator,False,Requires documented population source before rates go live.
3,Shooting definition,False,Requires field-value semantics and offense-vs-report decision.
4,Current response pipeline reproduced,True,Notebook event IDs should exactly match current production logic.
5,Dynamic map-coverage metric,True,Use filtered analytical population; do not exclude unmappable offenses from other metrics.



CURRENT METHODOLOGY CANDIDATES
Crime counting unit: unique offense_id
Previous comparison: immediately preceding equal-length period
Neighborhood crime ranking: unique offense volume
Response-time counting unit: unique CAD event
Candidate response priorities: [1, 2, 3]
Current candidate max response: 1440 minutes
Candidate minimum neighborhood response sample: 30

Current example unmappable share: 16.25%

Report-before-offense timestamp anomalies from the prior QA do not affect ordinary offense-count metrics; they remain relevant if reporting-lag analysis is introduced later.
